# What is KITTI for annotation?

Object detection datasets can use the KITTI format to assign bounding boxes on objects detected within a set of images. Each numbered image has a corresponding label file. The label file contains a 15 element row of data, for each object detected. The 15 elements correspond to:
 
![KITTI_table.png](./img/KITTI_table.png)

Detailed information on the KITTI format can found here: https://docs.nvidia.com/metropolis/TLT/tlt-user-guide/text/data_annotation_format.html

## KITTI Bounding Box Example
Let's draw a bounding box in one of our images from the training set

In [ ]:
image = 'data_sandbox\\training\\image\\vlcsnap-2022-08-06-16h07m17s914.jpg'
image_path = 'data_sandbox\\training\\image\\'
label = 'data_sandbox\\training\\label\\vlcsnap-2022-08-06-16h07m17s914.txt'
label_path = 'data_sandbox\\training\\label\\'

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Load the JPG image as a NumPy array
img = mpimg.imread(image)

# Display the image
plt.imshow(img)
plt.axis("off")  # Hide grid lines and pixel axis coordinates
plt.show()

In [ ]:
import os
import argparse
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

CLASS_COLORS = {
    "plastic_bag":        (255,   0,   0),   # Red   (B,G,R) for OpenCV
    "placeholder1": (0, 255,   0),   # Green
    "placeholder2":    (0,   0, 255),   # Blue
    # default colour when class not listed above
    "default":    (255, 255,   0),   # Cyan
}

def parse_label_file(label_path: Path):
    """
    Read a single KITTI label file and return a list of dictionaries:
        [
            {"type": "Car", "bbox": (xmin, ymin, xmax, ymax)},
            ...
        ]
    """
    objects = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:          # malformed line – skip
                continue
            obj_type = parts[0]
            # bbox coordinates are at index 4‑7 (0‑based)
            xmin = float(parts[4])
            ymin = float(parts[5])
            xmax = float(parts[6])
            ymax = float(parts[7])
            objects.append({"type": obj_type,
                            "bbox": (xmin, ymin, xmax, ymax)})
    return objects

In [ ]:
def draw_boxes(img: np.ndarray, objects: list):
    """
    Draw bounding boxes on an image (OpenCV format BGR).
    Returns the same image with boxes drawn in‑place.
    """
    for obj in objects:
        cls = obj["type"]
        xmin, ymin, xmax, ymax = map(int, obj["bbox"])
        color = CLASS_COLORS.get(cls, CLASS_COLORS["default"])
        # rectangle
        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), color, thickness=2)
        # label text (class name)
        label = cls
        (tw, th), _ = cv2.getTextSize(label,
                                      cv2.FONT_HERSHEY_SIMPLEX,
                                      fontScale=0.6,
                                      thickness=1)
        # background for readability
        cv2.rectangle(img,
                      (xmin, ymin - th - 4),
                      (xmin + tw, ymin),
                      color,
                      thickness=-1)  # filled
        # put text
        cv2.putText(img,
                    label,
                    (xmin, ymin - 2),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=0.6,
                    color=(0, 0, 0),  # black text
                    thickness=1,
                    lineType=cv2.LINE_AA)
    return img

In [ ]:
def plot_bbox(img_dir: Path, label_dir: Path, out_dir: Path = None, show: bool = True):
    img_dir = img_dir.expanduser()
    label_dir = label_dir.expanduser()
    if out_dir:
        out_dir = out_dir.expanduser()
        out_dir.mkdir(parents=True, exist_ok=True)

    # Accept typical image extensions
    img_exts = {".png", ".jpg", ".jpeg", ".bmp"}

    # Loop over every label file
    for label_path in sorted(label_dir.glob("*.txt")):
        base_name = label_path.stem          # e.g. "000001"
        # Find the image with the same stem
        img_path = None
        for ext in img_exts:
            candidate = img_dir / f"{base_name}{ext}"
            if candidate.is_file():
                img_path = candidate
                break
        if img_path is None:
            print(f"[WARN] No image found for label {label_path.name}")
            continue

        # Load image (OpenCV reads as BGR)
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"[ERROR] Failed to read image {img_path}")
            continue

        # Parse annotations & draw them
        objects = parse_label_file(label_path)
        img_annot = draw_boxes(img.copy(), objects)

        # Show with Matplotlib (converts BGR → RGB)
        if show:
            plt.figure(figsize=(10, 6))
            plt.imshow(cv2.cvtColor(img_annot, cv2.COLOR_BGR2RGB))
            plt.title(f"{base_name} – {len(objects)} objects")
            plt.axis("off")
            plt.tight_layout()
            plt.show()

        # Optional: save the annotated image
        if out_dir:
            out_path = out_dir / f"{base_name}_annotated{img_path.suffix}"
            cv2.imwrite(str(out_path), img_annot)
            print(f"[INFO] Saved annotated image to {out_path}")

In [ ]:
plot_bbox(img_dir=Path(image_path),
         label_dir=Path(label_path))